**Ejercicio 1**: Implemente un algoritmo de optimizacion por enjambre de partıculas
y utilıcelo para encontrar el mınimo global de las funciones del Ejercicio 1 de
la Guıa de trabajos practicos 6.
Compare los resultados en relacion a los obtenidos con algoritmos geneticos,
en terminos de las soluciones encontradas y la velocidad de convergencia.

In [2]:
import numpy as np
import time
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

In [3]:
def crear_enjambre(n_particles, min, max): 
  # Creamos un enjambre de particulas
  particulas = np.zeros((n_particles, 1))

  for i in range(n_particles):
      particulas[i] = np.random.uniform(min, max)

  return particulas 

In [4]:
def fitness_function_f1(x):
  return -(x * np.sin(np.sqrt(np.abs(x)))) # -f1 para obtener la aptitud mínima en vez de máxima


In [5]:
def pso(fitness_func, min, max, c1= 0.1, c2 = 0.1, n_particles=10, max_iter=100):
  inicio = time.time()
  x = np.random.uniform(min, max, n_particles)
  v = np.zeros_like(x)

  best_local = x.copy()

  best_local_val = np.array([fitness_func(p) for p in x])

  best_global = best_local[np.argmin(best_local_val)]

  best_global_val = fitness_func(best_global)

  history = [best_global_val]
  iter = 0 
  while iter < max_iter: 
    r1, r2 = np.random.rand(), np.random.rand()
    for k in range(n_particles): 
      v[k] +=  (c1 * r1 * (best_local[k] - x[k])) + (c2 * r2 * (best_global - x[k]))
      x[k] += v[k]
      x[k] = np.clip(x[k], min, max)

    for k in range(n_particles): 
      val = fitness_func(x[k])
      if val < best_local_val[k]: 
        best_local[k] = x[k]
        best_local_val[k] = val
      if val < best_global_val:
        best_global = x[k] 
        best_global_val = val

    
    iter+=1
    
  fin = time.time()
  print(f"Mejor valor encontrado: {best_global_val:.6f}")
  print(f"En posición: {best_global:.6f}")
  print(f"El algoritmo terminó en {round(fin - inicio, 4)} segundos")
  

In [6]:
pso(fitness_function_f1, -512, 512)


Mejor valor encontrado: -418.982300
En posición: 421.036956
El algoritmo terminó en 0.0287 segundos


In [7]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import time

def fitness_function_f1(x):
    return -x * np.sin(np.sqrt(abs(x)))

def pso_gif(fitness_func, xmin, xmax, c1=0.5, c2=0.5, n_particles=15, max_iter=50):
    inicio = time.time()
    
    # Inicialización
    x = np.random.uniform(xmin, xmax, n_particles)
    v = np.zeros_like(x)
    best_local = x.copy()
    best_local_val = np.array([fitness_func(p) for p in x])
    best_global = best_local[np.argmin(best_local_val)]
    best_global_val = fitness_func(best_global)

    # Guardamos historia para animar
    swarm_history = [x.copy()]
    
    # Bucle principal
    for _ in range(max_iter):
        r1, r2 = np.random.rand(), np.random.rand()
        for k in range(n_particles):
            v[k] += (c1 * r1 * (best_local[k] - x[k])) + (c2 * r2 * (best_global - x[k]))
            x[k] += v[k]
            x[k] = np.clip(x[k], xmin, xmax)

            val = fitness_func(x[k])
            if val < best_local_val[k]:
                best_local[k] = x[k]
                best_local_val[k] = val
            if val < best_global_val:
                best_global = x[k]
                best_global_val = val

        swarm_history.append(x.copy())

    fin = time.time()
    print(f"Mejor valor encontrado: {best_global_val:.6f}")
    print(f"En posición: {best_global:.6f}")
    print(f"El algoritmo terminó en {round(fin - inicio, 4)} segundos")

    # =======================
    # 🎞️ Animación del enjambre
    # =======================
    X = np.linspace(xmin, xmax, 1000)
    Y = fitness_func(X)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(X, Y, 'gray', lw=2, label="f(x)")
    scatter = ax.scatter([], [], color='red', s=50, label="Partículas")
    best_dot, = ax.plot([], [], 'bo', markersize=8, label="Mejor global")

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(min(Y)*1.1, max(Y)*1.1)
    ax.legend()
    ax.set_title("Optimización por enjambre de partículas")

    def init():
        scatter.set_offsets(np.c_[[], []])
        best_dot.set_data([], [])
        return scatter, best_dot

    def update(frame):
        x_pos = swarm_history[frame]
        y_pos = [fitness_func(xi) for xi in x_pos]
        scatter.set_offsets(np.c_[x_pos, y_pos])
        # ✅ Usar listas para el punto azul
        best_dot.set_data([best_global], [fitness_func(best_global)])
        ax.set_xlabel(f"Iteración {frame+1}/{max_iter}")
        return scatter, best_dot


    ani = FuncAnimation(fig, update, frames=len(swarm_history),
                        init_func=init, blit=True, interval=1000)

    ani.save("pso_enjambre.gif", writer=PillowWriter(fps=10))
    plt.close(fig)
    print("✅ GIF guardado como 'pso_enjambre.gif'")

    return best_global, best_global_val, "pso_enjambre.gif"


In [9]:
best_x, best_val, gif = pso_gif(fitness_function_f1, -512, 512, c1=0.1, c2=0.1, n_particles=10, max_iter=50)


Mejor valor encontrado: -418.968377
En posición: 420.629614
El algoritmo terminó en 0.0036 segundos
✅ GIF guardado como 'pso_enjambre.gif'
